# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [5]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
# Bỏ mọi module src.* đã nạp: reload riêng augment không nạp lại extract/view/landmarks nó import,
# nên kernel chạy lâu sẽ giữ bản cũ (vd. ImportError PITCH_AUG_DEG).
for name in [m for m in sys.modules if m == 'src' or m.startswith('src.')]: del sys.modules[name]
reader = importlib.import_module('src.data.data_reader')
augment = importlib.import_module('src.data.augment')
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'  # 'all', 'xin chào', hoặc ['xin chào', 'cảm ơn']
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))  # N; -1 = tăng cường toàn bộ, không giới hạn
COPIES_PER_SAMPLE = 2    # chỉ dùng khi MIN_SAMPLES_PER_CLASS = -1
NOISE_STRENGTH = 1.0    # khoảng (0, 3]
PITCH_DEGREES = augment.PITCH_AUG_DEG  # mỗi bản sao lấy pitch camera ngẫu nhiên trong ±PITCH_DEGREES độ (cố định trên clip); khoảng [0, 45], 0 = không đổi pitch
SEED = None  # đặt số nguyên để chạy tái lập
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
if not 0 <= PITCH_DEGREES <= 45: raise ValueError('PITCH_DEGREES phải trong [0, 45]')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')


495 mẫu dự kiến tạo trong 30 lớp


## Kiểm tra trước khi tạo


In [6]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


a: hiện có 20, thêm 25, sau đó có 45
age: hiện có 30, thêm 15, sau đó có 45
b: hiện có 30, thêm 15, sau đó có 45
c: hiện có 30, thêm 15, sau đó có 45
cảm ơn: hiện có 30, thêm 15, sau đó có 45
dấu huyền: hiện có 30, thêm 15, sau đó có 45
dấu hỏi: hiện có 20, thêm 25, sau đó có 45
dấu nặng: hiện có 30, thêm 15, sau đó có 45
dấu sắc: hiện có 30, thêm 15, sau đó có 45
dấu â,ô,ê: hiện có 30, thêm 15, sau đó có 45
dấu ă: hiện có 20, thêm 25, sau đó có 45
dấu ơ,ư: hiện có 30, thêm 15, sau đó có 45
e: hiện có 30, thêm 15, sau đó có 45
g: hiện có 30, thêm 15, sau đó có 45
h: hiện có 30, thêm 15, sau đó có 45
i: hiện có 30, thêm 15, sau đó có 45
k: hiện có 30, thêm 15, sau đó có 45
l: hiện có 30, thêm 15, sau đó có 45
m: hiện có 30, thêm 15, sau đó có 45
n: hiện có 30, thêm 15, sau đó có 45
o: hiện có 30, thêm 15, sau đó có 45
r: hiện có 15, thêm 30, sau đó có 45
t: hiện có 30, thêm 15, sau đó có 45
ten_la: hiện có 30, thêm 15, sau đó có 45
toi: hiện có 30, thêm 15, sau đó có 45
u: hiện có 30, t

## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [7]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng, pitch_degrees=PITCH_DEGREES)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


[1/365] a/0 -> 20, 21
[2/365] a/1 -> 22, 23
[3/365] a/10 -> 24
[4/365] a/11 -> 25
[5/365] a/12 -> 26
[6/365] a/13 -> 27
[7/365] a/14 -> 28
[8/365] a/15 -> 29
[9/365] a/16 -> 30
[10/365] a/17 -> 31
[11/365] a/18 -> 32
[12/365] a/19 -> 33
[13/365] a/2 -> 34, 35
[14/365] a/3 -> 36, 37
[15/365] a/4 -> 38, 39
[16/365] a/5 -> 40
[17/365] a/6 -> 41
[18/365] a/7 -> 42
[19/365] a/8 -> 43
[20/365] a/9 -> 44
[21/365] age/0 -> 30
[22/365] age/1 -> 31
[23/365] age/10 -> 32
[24/365] age/11 -> 33
[25/365] age/12 -> 34
[26/365] age/13 -> 35
[27/365] age/14 -> 36
[28/365] age/2 -> 37
[29/365] age/3 -> 38
[30/365] age/4 -> 39
[31/365] age/5 -> 40
[32/365] age/6 -> 41
[33/365] age/7 -> 42
[34/365] age/8 -> 43
[35/365] age/9 -> 44
[36/365] b/0 -> 30, 31, 32
[37/365] b/1 -> 33, 34, 35
[38/365] b/2 -> 36, 37, 38
[39/365] b/3 -> 39, 40, 41
[40/365] b/4 -> 42, 43, 44
[41/365] c/0 -> 30
[42/365] c/1 -> 31
[43/365] c/10 -> 32
[44/365] c/11 -> 33
[45/365] c/12 -> 34
[46/365] c/13 -> 35
[47/365] c/14 -> 36
[48/36

## Kiểm tra group


In [8]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 1350
Tổng: {'a': 45, 'age': 45, 'b': 45, 'c': 45, 'cảm ơn': 45, 'dấu huyền': 45, 'dấu hỏi': 45, 'dấu nặng': 45, 'dấu sắc': 45, 'dấu â,ô,ê': 45, 'dấu ă': 45, 'dấu ơ,ư': 45, 'e': 45, 'g': 45, 'h': 45, 'i': 45, 'k': 45, 'l': 45, 'm': 45, 'n': 45, 'o': 45, 'r': 45, 't': 45, 'ten_la': 45, 'toi': 45, 'u': 45, 'v': 45, 'xin chào': 45, 'y': 45, 'địa chỉ': 45}
Group gốc độc lập: {'a': 20, 'age': 15, 'b': 5, 'c': 20, 'cảm ơn': 15, 'dấu huyền': 5, 'dấu hỏi': 20, 'dấu nặng': 15, 'dấu sắc': 5, 'dấu â,ô,ê': 5, 'dấu ă': 20, 'dấu ơ,ư': 5, 'e': 5, 'g': 15, 'h': 30, 'i': 15, 'k': 15, 'l': 5, 'm': 30, 'n': 15, 'o': 5, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 5, 'u': 15, 'v': 5, 'xin chào': 15, 'y': 16, 'địa chỉ': 15}
949 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
